# 02 — CRCP Benthic Explorer: Full Pipeline Demo

This notebook runs the complete data pipeline end-to-end:

1. **Ingest** — Query ERDDAP for point-level annotations
2. **Transform** — Aggregate points → images → site-year summaries
3. **Spatial** — Build a GeoDataFrame with validated coordinates
4. **Validate** — Run QA checks at each stage
5. **Export** — Save to GeoJSON for inspection or publishing

This notebook is designed to run both locally and as an ArcGIS Online
Hosted Notebook.

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

from src import ingest, transform, spatial, validate

## Step 1: Ingest from ERDDAP

In [ ]:
SOURCE_ID = "hawaii"

points_df = ingest.fetch_source(SOURCE_ID)
print(f"Fetched {len(points_df):,} point annotations")
print(f"Columns: {list(points_df.columns)}")
points_df.head(3)

### Validate raw points

In [ ]:
pt_report = validate.validate_points(points_df)
print(pt_report.summary())

## Step 2: Transform — Points to Images

In [ ]:
images_df = transform.points_to_images(points_df)
print(f"Image records: {len(images_df):,}")
print(f"Tier-1 cover columns: {[c for c in images_df.columns if c.startswith('t1_')]}")
images_df.head(3)

### Validate image-level data

In [ ]:
img_report = validate.validate_images(images_df)
print(img_report.summary())

## Step 3: Transform — Images to Site-Year Summaries

In [ ]:
sites_df = transform.images_to_sites(images_df)
print(f"Site-year records: {len(sites_df):,}")
print(f"Islands: {sorted(sites_df['island'].unique())}")
print(f"Years:   {sorted(sites_df['obs_year'].unique())}")
sites_df.head(3)

## Step 4: Build GeoDataFrame and validate coordinates

In [ ]:
gdf = spatial.build_site_geodataframe(sites_df)
coord_warnings = spatial.validate_coordinates(gdf, SOURCE_ID)

if coord_warnings:
    for w in coord_warnings:
        print(f"  WARNING: {w}")
else:
    print("All coordinates within expected bounds.")

site_report = validate.validate_sites(gdf, expected_images=len(images_df))
print(site_report.summary())

## Step 5: Preview and export

In [ ]:
# Quick summary statistics
print(f"Total sites:  {len(gdf):,}")
print(f"Total images: {gdf['n_images'].sum():,}")
print(f"\nMean coral cover by island:")
island_coral = gdf.groupby("island")["mean_t1_CORAL"].mean().sort_values(ascending=False)
for island, val in island_coral.items():
    print(f"  {island:20s} {val:6.1f}%")

In [ ]:
# Export to GeoJSON
from pathlib import Path

output_dir = Path("..") / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{SOURCE_ID}_site_summary.geojson"

gdf.to_file(output_path, driver="GeoJSON")
print(f"Exported to {output_path} ({output_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
# Simple matplotlib preview
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    gdf.plot(
        ax=ax,
        column="mean_t1_CORAL",
        cmap="YlOrRd",
        legend=True,
        legend_kwds={"label": "Mean Coral Cover (%)"},
        markersize=10,
        alpha=0.7,
    )
    ax.set_title(f"CRCP Benthic Cover — {SOURCE_ID.title()}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available — skipping preview plot.")